# NN7 step-zero architecture search

This notebook searches for a **step-zero branched-attention configuration that visibly changes the face relative to ordinary PhotoMaker without immediately recreating N3a's pose and artifact failures**.

It keeps the target images, prompts, seeds, PhotoMaker identity inputs, RealVis validation backbone, and fixed target bboxes constant. It changes only the branched-attention architecture and authority.

## Why N3a is included

The original N3a/NN1 mechanism is the positive control:

```text
target-face Q
    attends
reference-face K/V

face attention candidate = reference candidate
layer scope = all self-attention layers
reference memory = full noised/evolving spatial grid
protected PhotoMaker epsilon = absent
```

This produces a face that is clearly different from PhotoMaker at step zero, but it also gives unaligned reference pose, expression, hair, accessories, and occluders excessive authority.

The search therefore stays close to the original mechanism while testing the smallest repairs:

```text
freeze/disable branched cross-attention
pack/normalize the reference face ROI
restrict reference ownership to up blocks
restore an explicit target-face candidate
protect a target-owned boundary ring
anchor the final epsilon outside an inner face core
vary initial reference ownership and BA start time
```

## Safety rules

- This notebook never writes into `src/`, `train.py`, or repository configs.
- Every architecture change is a Hydra override or an in-memory processor mutation.
- All results go under:

```text
diffusion_template/Jul_new_exp/22Jul_debug/experiments/
```

- Every experiment gets its own folder with the exact recipe, compact model signature, metrics JSON/CSV, images, crops, difference maps, and contact sheet.
- The notebook fingerprints core Python files before and after every experiment and stops if they changed.
- No training or checkpoint loading is performed.


In [ ]:
# ---------------------------
# USER SETTINGS
# ---------------------------
from pathlib import Path
from datetime import datetime, timezone

REPO_ROOT = Path("/home/niko/rsrch/diffusion_template")
NOTEBOOK_DIR = REPO_ROOT / "Jul_new_exp/22Jul_debug"
EXPERIMENTS_ROOT = NOTEBOOK_DIR / "experiments"

BASE_MODEL = "SG161222/RealVisXL_V4.0"
PHOTOMAKER_PATH = Path("/home/niko/models/PhotoMaker-V2/photomaker-v2.bin")
MODEL_WEIGHT_DTYPE = "bf16"
MODEL_LORA_RANK = 32

# Exactly the first four records in the current manual validation dataset.
SAMPLE_INDICES = [0, 1, 2, 3]

# Fast screening schedule. Finalists should be repeated with 50 steps.
NUM_INFERENCE_STEPS = 20
PHOTOMAKER_START_STEP = 4     # 20% of the schedule
DEFAULT_BA_START_STEP = 6     # 30% of the schedule
GUIDANCE_SCALE = 5.0
REFERENCE_NOISE_SEED = 918273

NEGATIVE_PROMPT = (
    "lowres, text, error, cropped, worst quality, low quality, "
    "jpeg artifacts, signature, watermark, blurry, deformed face, "
    "duplicated face, extra eyes, extra mouth"
)

# "quick" runs the primary N3a-to-NN7 ladder.
# "extended" adds more one-axis ablations.
RUN_PROFILE = "quick"

# Set to a list of experiment IDs to run only those entries.
# Example: ["n3a_exact", "n3a_roi_up_core_ring_anchor"]
SELECTED_EXPERIMENT_IDS = None

# Optional extra diagnostics cost an additional ordinary U-Net prediction.
COLLECT_LIGHT_PROCESSOR_DIAGNOSTICS = False
SHOW_PROGRESS_BARS = True

# Screening thresholds. These are deliberately conservative heuristics,
# not publication metrics.
FACE_MAE_VISIBLE_MIN = 0.012
FACE_MAE_DESTRUCTIVE_MAX = 0.12
OUTSIDE_MAE_MAX = 0.015
LANDMARK_DISPLACEMENT_MAX = 0.08
BBOX_IOU_MIN = 0.60
REFERENCE_GAIN_MIN = 0.003

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Repository:", REPO_ROOT)
print("Artifacts:", EXPERIMENTS_ROOT)
print("Profile:", RUN_PROFILE)


In [ ]:
# ---------------------------
# ENVIRONMENT AND IMPORTS
# ---------------------------
import os
import sys
import gc
import math
import time
import shutil
import hashlib
import traceback
import subprocess
from contextlib import contextmanager
from copy import deepcopy
from dataclasses import dataclass, field, asdict
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display

assert REPO_ROOT.exists(), REPO_ROOT
assert PHOTOMAKER_PATH.exists(), PHOTOMAKER_PATH

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("FACEANALYSIS_CPU", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires one CUDA GPU.")

from accelerate import Accelerator
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.model.photomaker_branched.attn_processor_cleanest import (
    BranchedAttnProcessor,
)
from src.model.photomaker_branched.packed_residual_attn_processor import (
    PackedResidualBranchedAttnProcessor,
)
from src.model.photomaker_branched.insightface_package import (
    analyze_faces,
    create_face_analyzer,
)
from src.pipelines.br_pipeline_helpers import (
    reset_branched_generation_caches,
)

ACCELERATOR = Accelerator()
DEVICE = ACCELERATOR.device

def devices_match(left, right) -> bool:
    left = torch.device(left)
    right = torch.device(right)
    if left.type != right.type:
        return False
    if left.type != "cuda":
        return left == right
    left_index = torch.cuda.current_device() if left.index is None else left.index
    right_index = torch.cuda.current_device() if right.index is None else right.index
    return left_index == right_index

def git_text(*args: str) -> str:
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_ROOT,
        check=False,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()

GIT_COMMIT = git_text("rev-parse", "HEAD")
GIT_BRANCH = git_text("rev-parse", "--abbrev-ref", "HEAD")
GIT_STATUS_AT_START = git_text("status", "--porcelain")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.get_device_name(torch.cuda.current_device()))
print("Device:", DEVICE)
print("Git:", GIT_BRANCH, GIT_COMMIT)


## Core-code protection

The search is intentionally disposable. The notebook may be edited, interrupted, or deleted. The project implementation must not be altered by the search.

The next cell hashes the relevant Python/config source tree. Every experiment verifies the same fingerprint after cleanup.


In [ ]:
# ---------------------------
# CORE CODE IMMUTABILITY GUARD
# ---------------------------
PROTECTED_ROOTS = [
    REPO_ROOT / "src",
    REPO_ROOT / "train.py",
]

def protected_files() -> list[Path]:
    files: list[Path] = []
    for root in PROTECTED_ROOTS:
        if root.is_file():
            files.append(root)
        elif root.is_dir():
            files.extend(sorted(root.rglob("*.py")))
            files.extend(sorted(root.rglob("*.yaml")))
    return sorted(set(files))

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def core_fingerprint() -> dict[str, str]:
    return {
        str(path.relative_to(REPO_ROOT)): file_sha256(path)
        for path in protected_files()
    }

CORE_FINGERPRINT = core_fingerprint()

def assert_core_unchanged() -> None:
    current = core_fingerprint()
    if current != CORE_FINGERPRINT:
        changed = sorted(
            set(current) ^ set(CORE_FINGERPRINT)
            | {
                key
                for key in set(current) & set(CORE_FINGERPRINT)
                if current[key] != CORE_FINGERPRINT[key]
            }
        )
        raise RuntimeError(
            "Protected repository code changed during the notebook run: "
            + ", ".join(changed[:20])
        )

print("Protected files:", len(CORE_FINGERPRINT))


## Load the first four validation records

The notebook reads `datasets.val.manual_val` from the repository configuration. It uses the current RealVis-derived generation bboxes, which is appropriate because the step-zero search runs on RealVis.


In [ ]:
# ---------------------------
# VALIDATION DATASET
# ---------------------------
datasets_cfg = OmegaConf.load(
    REPO_ROOT / "src/configs/datasets/all_datasets.yaml"
)
manual_cfg = OmegaConf.create(
    OmegaConf.to_container(
        datasets_cfg.val.manual_val,
        resolve=True,
    )
)
manual_cfg.limit = None
manual_cfg.pop("subset_size", None)
manual_cfg.pop("subset_seed", None)

validation_dataset = instantiate(manual_cfg)

if max(SAMPLE_INDICES) >= len(validation_dataset):
    raise IndexError(
        f"Requested sample {max(SAMPLE_INDICES)} but dataset has "
        f"{len(validation_dataset)} records"
    )

validation_samples = []
for source_index in SAMPLE_INDICES:
    sample = validation_dataset[source_index]
    sample = dict(sample)
    sample["source_index"] = int(source_index)
    sample["reference_path"] = str(
        validation_dataset.samples[source_index]["image_path"]
    )
    if sample.get("face_bbox_ref") is None:
        raise RuntimeError(
            f"Validation sample {source_index} has no reference bbox"
        )
    if sample.get("face_bbox_gen") is None:
        raise RuntimeError(
            f"Validation sample {source_index} has no fixed RealVis generation bbox"
        )
    validation_samples.append(sample)

sample_table = pd.DataFrame(
    [
        {
            "source_index": sample["source_index"],
            "id": sample["id"],
            "seed": sample["seed"],
            "prompt": sample["prompt"],
            "reference_path": sample["reference_path"],
            "face_bbox_ref": sample["face_bbox_ref"],
            "face_bbox_gen": sample["face_bbox_gen"],
        }
        for sample in validation_samples
    ]
)
display(sample_table)

samples_manifest_path = EXPERIMENTS_ROOT / f"{RUN_ID}__validation_samples.json"
samples_manifest_path.write_text(
    json.dumps(
        sample_table.to_dict(orient="records"),
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)
print("Saved sample manifest:", samples_manifest_path)


In [ ]:
# ---------------------------
# FACE ANALYSIS AND METRICS
# ---------------------------
face_analyzer = create_face_analyzer(
    providers=["CPUExecutionProvider"],
    allowed_modules=["detection", "recognition"],
    ctx_id=-1,
    det_size=(640, 640),
    fallback_ctx_id=-1,
    quiet=True,
)

def face_value(face, key: str):
    if isinstance(face, dict):
        return face.get(key)
    return getattr(face, key, None)

def detect_largest_face(image: Image.Image) -> dict[str, Any]:
    array_bgr = np.asarray(image.convert("RGB"))[:, :, ::-1]
    faces = analyze_faces(face_analyzer, array_bgr)
    if not faces:
        return {
            "detected": False,
            "bbox": None,
            "embedding": None,
            "kps": None,
            "det_score": float("nan"),
        }

    def area(face) -> float:
        bbox = face_value(face, "bbox")
        if bbox is None:
            return -1.0
        x0, y0, x1, y1 = (float(value) for value in bbox)
        return max(0.0, x1 - x0) * max(0.0, y1 - y0)

    selected = max(faces, key=area)
    bbox = face_value(selected, "bbox")
    embedding = face_value(selected, "embedding")
    kps = face_value(selected, "kps")
    det_score = face_value(selected, "det_score")

    embedding_out = None
    if embedding is not None:
        embedding_out = np.asarray(embedding, dtype=np.float32)
        norm = np.linalg.norm(embedding_out)
        if norm > 0:
            embedding_out = embedding_out / norm

    return {
        "detected": bbox is not None,
        "bbox": [float(v) for v in bbox] if bbox is not None else None,
        "embedding": embedding_out,
        "kps": (
            np.asarray(kps, dtype=np.float32)
            if kps is not None
            else None
        ),
        "det_score": float(det_score) if det_score is not None else float("nan"),
    }

def cosine_np(left, right) -> float:
    if left is None or right is None:
        return float("nan")
    denom = float(np.linalg.norm(left) * np.linalg.norm(right))
    if denom <= 0:
        return float("nan")
    return float(np.dot(left, right) / denom)

def image_array(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0

def clipped_bbox(bbox, image: Image.Image) -> list[int]:
    x0, y0, x1, y1 = (float(v) for v in bbox)
    return [
        max(0, min(image.width, int(math.floor(x0)))),
        max(0, min(image.height, int(math.floor(y0)))),
        max(0, min(image.width, int(math.ceil(x1)))),
        max(0, min(image.height, int(math.ceil(y1)))),
    ]

def crop_by_bbox(image: Image.Image, bbox) -> Image.Image:
    x0, y0, x1, y1 = clipped_bbox(bbox, image)
    return image.crop((x0, y0, x1, y1))

def bbox_iou(left, right) -> float:
    if left is None or right is None:
        return float("nan")
    ax0, ay0, ax1, ay1 = map(float, left)
    bx0, by0, bx1, by1 = map(float, right)
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    area_a = max(0.0, ax1 - ax0) * max(0.0, ay1 - ay0)
    area_b = max(0.0, bx1 - bx0) * max(0.0, by1 - by0)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else float("nan")

def normalized_landmark_displacement(
    current: dict[str, Any],
    baseline: dict[str, Any],
) -> float:
    current_kps = current.get("kps")
    baseline_kps = baseline.get("kps")
    baseline_bbox = baseline.get("bbox")
    if (
        current_kps is None
        or baseline_kps is None
        or baseline_bbox is None
        or current_kps.shape != baseline_kps.shape
    ):
        return float("nan")
    x0, y0, x1, y1 = map(float, baseline_bbox)
    diagonal = math.hypot(x1 - x0, y1 - y0)
    if diagonal <= 0:
        return float("nan")
    return float(
        np.sqrt(np.mean((current_kps - baseline_kps) ** 2))
        / diagonal
    )

def region_metrics(
    image: Image.Image,
    baseline: Image.Image,
    bbox,
) -> dict[str, float]:
    current = image_array(image)
    base = image_array(baseline)
    difference = np.abs(current - base)

    x0, y0, x1, y1 = clipped_bbox(bbox, image)
    face_mask = np.zeros(current.shape[:2], dtype=np.float32)
    face_mask[y0:y1, x0:x1] = 1.0

    width = max(2, int(round(0.05 * max(1, min(x1 - x0, y1 - y0)))))
    outer = np.zeros_like(face_mask)
    ox0, oy0 = max(0, x0 - width), max(0, y0 - width)
    ox1, oy1 = min(image.width, x1 + width), min(image.height, y1 + width)
    outer[oy0:oy1, ox0:ox1] = 1.0
    inner = np.zeros_like(face_mask)
    ix0, iy0 = min(x1, x0 + width), min(y1, y0 + width)
    ix1, iy1 = max(x0, x1 - width), max(y0, y1 - width)
    if ix1 > ix0 and iy1 > iy0:
        inner[iy0:iy1, ix0:ix1] = 1.0
    ring = np.clip(outer - inner, 0.0, 1.0)

    def masked_mae(mask: np.ndarray) -> float:
        count = max(float(mask.sum() * current.shape[2]), 1.0)
        return float((difference * mask[:, :, None]).sum() / count)

    changed = np.any(
        np.asarray(image.convert("RGB"))
        != np.asarray(baseline.convert("RGB")),
        axis=2,
    )

    return {
        "full_mae_vs_pm": float(difference.mean()),
        "face_mae_vs_pm": masked_mae(face_mask),
        "outside_mae_vs_pm": masked_mae(1.0 - face_mask),
        "boundary_ring_mae_vs_pm": masked_mae(ring),
        "changed_pixel_fraction": float(changed.mean()),
        "exact_rgb_equal_to_pm": bool(not changed.any()),
    }

def tensor_or_image_hash(image: Image.Image) -> str:
    return hashlib.sha256(
        np.asarray(image.convert("RGB")).tobytes()
    ).hexdigest()

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


## Candidate architecture ladder

The quick suite starts from exact N3a, then removes one failure source at a time while preserving the central target-Q → reference-K/V idea.

The modern NN7a variants are included as low-change controls, not as the main search direction.


In [ ]:
# ---------------------------
# EXPERIMENT SPECIFICATIONS
# ---------------------------
@dataclass(frozen=True)
class ExperimentSpec:
    experiment_id: str
    family: str
    role: str
    base_config: str
    description: str
    overrides: dict[str, Any] = field(default_factory=dict)
    runtime_mutations: dict[str, Any] = field(default_factory=dict)
    generation: dict[str, Any] = field(default_factory=dict)
    tags: tuple[str, ...] = ()

def spec(
    experiment_id: str,
    *,
    family: str,
    role: str,
    base_config: str,
    description: str,
    overrides: Optional[dict[str, Any]] = None,
    runtime_mutations: Optional[dict[str, Any]] = None,
    generation: Optional[dict[str, Any]] = None,
    tags: tuple[str, ...] = (),
) -> ExperimentSpec:
    return ExperimentSpec(
        experiment_id=experiment_id,
        family=family,
        role=role,
        base_config=base_config,
        description=description,
        overrides=overrides or {},
        runtime_mutations=runtime_mutations or {},
        generation=generation or {},
        tags=tags,
    )

N3A_BASE = "one_id_ba_NN1a_n3a_replay"
NN7_V1_BASE = "one_id_ba_NN7a_init"
NN7_V2_BASE = "one_id_ba_NN7a_init_v2"

QUICK_EXPERIMENTS = [
    spec(
        "n3a_exact",
        family="legacy_spatial_ba",
        role="positive_control",
        base_config=N3A_BASE,
        description=(
            "Exact guarded N3a replay: full reference grid, reference-only "
            "target face, all self-attention layers, branched CA enabled, "
            "no protected epsilon anchor."
        ),
        overrides={
            "disable_branched_ca": False,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "full_grid",
            "model.ba_sa_face_mode": "reference",
            "model.ba_sa_ref_layer_scope": "all",
            "model.ba_output_anchor_mode": "none",
        },
        tags=("original", "strong", "unsafe-control"),
    ),
    spec(
        "n3a_no_ca",
        family="legacy_spatial_ba",
        role="first_repair",
        base_config=N3A_BASE,
        description=(
            "Original N3a self-attention with branched cross-attention disabled. "
            "Tests the historically cleanest immediate repair."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "full_grid",
            "model.ba_sa_face_mode": "reference",
            "model.ba_sa_ref_layer_scope": "all",
            "model.ba_output_anchor_mode": "none",
        },
        tags=("original-q-ref-kv", "ca-off"),
    ),
    spec(
        "n3a_roi_all_reference",
        family="legacy_spatial_ba",
        role="token_repair",
        base_config=N3A_BASE,
        description=(
            "N3a reference-only ownership at all SA layers, but uses a dense "
            "normalized face ROI instead of a masked full grid; branched CA off."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 8,
            "model.ba_sa_face_mode": "reference",
            "model.ba_sa_ref_layer_scope": "all",
            "model.ba_output_anchor_mode": "none",
        },
        tags=("original-q-ref-kv", "roi", "ca-off"),
    ),
    spec(
        "n3a_roi_up_reference_anchor",
        family="legacy_spatial_ba",
        role="layer_and_output_repair",
        base_config=N3A_BASE,
        description=(
            "Reference-only face attention remains strong, but only in up "
            "blocks; normalized ROI, CA off, and ordinary PhotoMaker epsilon "
            "outside the inner core."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 8,
            "model.ba_sa_face_mode": "reference",
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_target_core_erode_frac": 0.10,
            "model.ba_output_anchor_mode": "base_outside_core",
        },
        tags=("original-q-ref-kv", "roi", "up-only", "anchor"),
    ),
    spec(
        "n3a_roi_up_core_ring_anchor",
        family="legacy_spatial_ba",
        role="primary_candidate",
        base_config=N3A_BASE,
        description=(
            "Closest strong repaired N3a candidate: reference owns an inner "
            "elliptical face core, target owns the surrounding face ring; "
            "normalized ROI, up blocks only, CA off, protected epsilon outside."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 8,
            "model.ba_sa_face_mode": "core_ring",
            "model.ba_sa_core_ratio": 0.68,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_target_core_erode_frac": 0.10,
            "model.ba_output_anchor_mode": "base_outside_core",
        },
        tags=("original-q-ref-kv", "core-ring", "up-only", "anchor"),
    ),
    spec(
        "n3a_roi_up_dual75_anchor",
        family="legacy_spatial_ba",
        role="primary_candidate",
        base_config=N3A_BASE,
        description=(
            "Separate target and reference face-attention candidates with "
            "75% initial reference ownership per head; normalized ROI, up "
            "blocks only, CA off, protected epsilon outside."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 8,
            "model.ba_sa_face_mode": "dual",
            "model.ba_sa_mix_init": 0.75,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_target_core_erode_frac": 0.10,
            "model.ba_output_anchor_mode": "base_outside_core",
        },
        tags=("original-q-ref-kv", "target-fallback", "ref-75", "anchor"),
    ),
    spec(
        "n3a_roi_up_confidence50_anchor",
        family="legacy_spatial_ba",
        role="primary_candidate",
        base_config=N3A_BASE,
        description=(
            "Target attention plus a confidence-weighted 50% reference "
            "residual; normalized ROI, up blocks only, CA off, protected "
            "epsilon outside."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 8,
            "model.ba_sa_face_mode": "confidence_residual",
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_target_core_erode_frac": 0.10,
            "model.ba_output_anchor_mode": "base_outside_core",
        },
        runtime_mutations={
            "legacy_confidence_gain": 0.50,
        },
        tags=("original-q-ref-kv", "confidence", "target-fallback", "anchor"),
    ),
    spec(
        "n3a_fullgrid_up_core_ring_anchor",
        family="legacy_spatial_ba",
        role="minimal_change_candidate",
        base_config=N3A_BASE,
        description=(
            "Keeps N3a's full-grid reference memory but limits it to an inner "
            "core in up blocks, disables CA, and anchors epsilon outside. This "
            "isolates how much ROI normalization matters."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "full_grid",
            "model.ba_sa_face_mode": "core_ring",
            "model.ba_sa_core_ratio": 0.68,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_target_core_erode_frac": 0.10,
            "model.ba_output_anchor_mode": "base_outside_core",
        },
        tags=("closest-to-n3a", "full-grid", "core-ring", "anchor"),
    ),
    spec(
        "nn7a_init_v1_default",
        family="clean_patch_ba",
        role="low_change_control",
        base_config=NN7_V1_BASE,
        description=(
            "Current partial sibling-K/V warm start with approximately 5% "
            "post-cap ownership at up1."
        ),
        tags=("modern-control", "clean-patches"),
    ),
    spec(
        "nn7a_init_v2_default",
        family="clean_patch_ba",
        role="low_change_control",
        base_config=NN7_V2_BASE,
        description=(
            "Complete sibling-attn2-space warm candidate, pre-cap gate, "
            "10% initial ownership and 20% final cap at up1."
        ),
        tags=("modern-control", "clean-patches", "full-attn2-space"),
    ),
    spec(
        "nn7a_init_v2_strong25",
        family="clean_patch_ba",
        role="modern_strong_control",
        base_config=NN7_V2_BASE,
        description=(
            "NN7a_init-v2 with 25% initial clean-patch ownership, a 35% cap, "
            "and a wider 9x9 local window. It tests whether current topology "
            "can leave the PhotoMaker basin at step zero."
        ),
        overrides={
            "model.ba_spatial_gate_max": 1.0,
            "model.ba_gate_init_logit": -1.0986122886681098,
            "model.ba_spatial_delta_rms_cap": 0.35,
            "model.ba_total_delta_rms_cap": 0.35,
            "model.ba_spatial_local_window": 9,
        },
        tags=("modern-control", "clean-patches", "strong"),
    ),
]

EXTENDED_EXPERIMENTS = [
    spec(
        "n3a_roi_up_dual50_anchor",
        family="legacy_spatial_ba",
        role="authority_ablation",
        base_config=N3A_BASE,
        description="Dual target/reference candidate with 50% reference ownership.",
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 8,
            "model.ba_sa_face_mode": "dual",
            "model.ba_sa_mix_init": 0.50,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_target_core_erode_frac": 0.10,
            "model.ba_output_anchor_mode": "base_outside_core",
        },
    ),
    spec(
        "n3a_roi_up_dual85_anchor",
        family="legacy_spatial_ba",
        role="authority_ablation",
        base_config=N3A_BASE,
        description="Dual target/reference candidate with 85% reference ownership.",
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 8,
            "model.ba_sa_face_mode": "dual",
            "model.ba_sa_mix_init": 0.85,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_target_core_erode_frac": 0.10,
            "model.ba_output_anchor_mode": "base_outside_core",
        },
    ),
    spec(
        "n3a_roi_up_core50_anchor",
        family="legacy_spatial_ba",
        role="core_size_ablation",
        base_config=N3A_BASE,
        description="Core-ring mode with a conservative 50% inner-core radius.",
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_face_mode": "core_ring",
            "model.ba_sa_core_ratio": 0.50,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_output_anchor_mode": "base_outside_core",
        },
    ),
    spec(
        "n3a_roi_up_core82_anchor",
        family="legacy_spatial_ba",
        role="core_size_ablation",
        base_config=N3A_BASE,
        description="Core-ring mode with an aggressive 82% inner-core radius.",
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_face_mode": "core_ring",
            "model.ba_sa_core_ratio": 0.82,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_output_anchor_mode": "base_outside_core",
        },
    ),
    spec(
        "n3a_roi12_up_core_ring_anchor",
        family="legacy_spatial_ba",
        role="roi_resolution_ablation",
        base_config=N3A_BASE,
        description="Core-ring candidate with a denser 12x12 normalized ROI.",
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_roi_grid_size": 12,
            "model.ba_sa_face_mode": "core_ring",
            "model.ba_sa_core_ratio": 0.68,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_output_anchor_mode": "base_outside_core",
        },
    ),
    spec(
        "n3a_roi_up_dual75_strict",
        family="legacy_spatial_ba",
        role="routing_ablation",
        base_config=N3A_BASE,
        description=(
            "Dual-75 candidate with strict face removal from the target "
            "background branch. This may improve isolation or damage continuity."
        ),
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": True,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_face_mode": "dual",
            "model.ba_sa_mix_init": 0.75,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_output_anchor_mode": "base_outside_core",
        },
    ),
    spec(
        "n3a_roi_up_core_ring_late",
        family="legacy_spatial_ba",
        role="schedule_ablation",
        base_config=N3A_BASE,
        description="Core-ring candidate starting later at 45% of denoising.",
        overrides={
            "disable_branched_ca": True,
            "strict_face_routing": False,
            "model.ba_sa_ref_token_mode": "roi",
            "model.ba_sa_face_mode": "core_ring",
            "model.ba_sa_core_ratio": 0.68,
            "model.ba_sa_ref_layer_scope": "up",
            "model.ba_output_anchor_mode": "base_outside_core",
        },
        generation={
            "branched_attn_start_step": 9,
        },
    ),
    spec(
        "nn7a_init_v2_upall25",
        family="clean_patch_ba",
        role="site_ablation",
        base_config=NN7_V2_BASE,
        description=(
            "Strong clean-patch takeover at both up0 and up1. Counterfactual "
            "training validation is disabled because this is a step-zero screen."
        ),
        overrides={
            "model.ba_site_policy": "up_blocks_attn1",
            "model.ba_spatial_site_policy": "up_blocks_attn1",
            "model.ba_spatial_gate_max": 1.0,
            "model.ba_gate_init_logit": -1.0986122886681098,
            "model.ba_spatial_delta_rms_cap": 0.35,
            "model.ba_total_delta_rms_cap": 0.35,
            "model.ba_spatial_local_window": 9,
        },
    ),
]

EXPERIMENT_SPECS = list(QUICK_EXPERIMENTS)
if RUN_PROFILE == "extended":
    EXPERIMENT_SPECS.extend(EXTENDED_EXPERIMENTS)
elif RUN_PROFILE != "quick":
    raise ValueError("RUN_PROFILE must be 'quick' or 'extended'")

if SELECTED_EXPERIMENT_IDS is not None:
    selected = set(SELECTED_EXPERIMENT_IDS)
    EXPERIMENT_SPECS = [
        item for item in EXPERIMENT_SPECS
        if item.experiment_id in selected
    ]
    missing = selected - {item.experiment_id for item in EXPERIMENT_SPECS}
    if missing:
        raise KeyError(f"Unknown experiment IDs: {sorted(missing)}")

spec_table = pd.DataFrame(
    [
        {
            "experiment_id": item.experiment_id,
            "family": item.family,
            "role": item.role,
            "base_config": item.base_config,
            "description": item.description,
        }
        for item in EXPERIMENT_SPECS
    ]
)
display(spec_table)
print("Experiments selected:", len(EXPERIMENT_SPECS))


## Model construction and in-memory mutation

Each experiment creates a fresh step-zero model and pipeline. No checkpoint is loaded.

The notebook reconstructs the exact recipe from the base Hydra config plus explicit overrides and writes that compact recipe to the experiment folder.


In [ ]:
# ---------------------------
# CONFIG / MODEL / PIPELINE HELPERS
# ---------------------------
def omega_update(cfg, dotted_path: str, value: Any) -> None:
    OmegaConf.update(
        cfg,
        dotted_path,
        value,
        merge=False,
        force_add=True,
    )

def compose_experiment_config(item: ExperimentSpec):
    GlobalHydra.instance().clear()
    with initialize_config_dir(
        version_base=None,
        config_dir=str(REPO_ROOT / "src/configs"),
    ):
        cfg = compose(config_name=item.base_config)

    common = {
        "model.pretrained_model_name_or_path": BASE_MODEL,
        "pipeline.pretrained_model_name_or_path": BASE_MODEL,
        "model.photomaker_path": str(PHOTOMAKER_PATH),
        "model.weight_dtype": MODEL_WEIGHT_DTYPE,
        "model.rank": int(MODEL_LORA_RANK),
        "model.num_inference_steps": int(NUM_INFERENCE_STEPS),
        "model.photomaker_start_step": int(PHOTOMAKER_START_STEP),
        "model.branched_attn_start_step": int(DEFAULT_BA_START_STEP),
        "model.ba_counterfactual_enabled": False,
        "model.use_id_loss": False,
        "model.ba_collect_aux_losses": False,
        "pretrained_model_for_validation_name_or_path": None,
        "update_proc_weights_val": True,
        "automatic_bboxes": False,
        "automatic_bboxes_every_val": False,
        "validation_args.num_images_per_prompt": 1,
        "validation_args.num_inference_steps": int(NUM_INFERENCE_STEPS),
        "validation_args.guidance_scale": float(GUIDANCE_SCALE),
        "pipeline.variant": None,
    }
    for path, value in common.items():
        omega_update(cfg, path, value)
    for path, value in item.overrides.items():
        omega_update(cfg, path, value)

    # Disabling branched CA must also disable its trainable processor route.
    # The legacy strict-install manifest otherwise still expects CA Q/K/V
    # parameters even though no CA processors were installed.
    if bool(OmegaConf.select(cfg, "disable_branched_ca", default=False)):
        omega_update(cfg, "train_branched_ca_lora", False)
        omega_update(cfg, "model.train_branched_ca_lora", False)

    OmegaConf.resolve(cfg)
    return cfg

def ba_runtime_kwargs(cfg) -> dict[str, Any]:
    keys = (
        "train_ba_only",
        "ba_train_top_k",
        "ba_patch_top_k",
        "non_ba_train",
        "train_ba_all_steps",
        "ba_weights_split",
        "use_attn_v2",
    )
    return {
        key: getattr(cfg, key)
        for key in keys
        if hasattr(cfg, key)
    }

def branched_registry(pipeline) -> dict[str, Any]:
    registry = getattr(pipeline, "_branched_attn_processors", None)
    if isinstance(registry, dict):
        return registry
    return dict(pipeline.unet.attn_processors)

def apply_runtime_mutations(
    model,
    pipeline,
    mutations: dict[str, Any],
) -> None:
    processors = [
        processor
        for processor in branched_registry(pipeline).values()
        if bool(getattr(processor, "_is_branched_processor", False))
    ]

    if "legacy_confidence_gain" in mutations:
        gain = float(mutations["legacy_confidence_gain"])
        if not -1.0 < gain < 1.0:
            raise ValueError("legacy_confidence_gain must be in (-1, 1)")
        raw = math.atanh(gain)
        found = 0
        for processor in processors:
            parameter = getattr(processor, "face_residual_gain", None)
            if parameter is not None:
                parameter.data.fill_(raw)
                found += 1
        if found == 0:
            raise RuntimeError(
                "No confidence-residual processors accepted the requested gain"
            )

    if "legacy_dual_weight" in mutations:
        weight = float(mutations["legacy_dual_weight"])
        if not 0.0 < weight < 1.0:
            raise ValueError("legacy_dual_weight must be in (0, 1)")
        raw = math.log(weight / (1.0 - weight))
        found = 0
        for processor in processors:
            parameter = getattr(processor, "face_mix_logits", None)
            if parameter is not None:
                parameter.data.fill_(raw)
                found += 1
        if found == 0:
            raise RuntimeError("No dual processors accepted the requested weight")

    if "legacy_scale" in mutations:
        value = float(mutations["legacy_scale"])
        for processor in processors:
            if isinstance(processor, BranchedAttnProcessor):
                processor.scale = value

    if "force_binary_masks" in mutations:
        value = bool(mutations["force_binary_masks"])
        for processor in processors:
            if hasattr(processor, "force_binary_masks"):
                processor.force_binary_masks = value

    if "packed_runtime_scale" in mutations:
        value = float(mutations["packed_runtime_scale"])
        for processor in processors:
            if isinstance(processor, PackedResidualBranchedAttnProcessor):
                processor.runtime_scale = value

def architecture_signature(model, pipeline) -> dict[str, Any]:
    rows = []
    for name, processor in sorted(branched_registry(pipeline).items()):
        if not bool(getattr(processor, "_is_branched_processor", False)):
            continue
        row = {
            "name": name,
            "class": processor.__class__.__name__,
            "kind": getattr(processor, "_branched_kind", None),
        }
        for field_name in (
            "ba_sa_ref_token_mode",
            "ba_sa_face_mode",
            "ba_sa_ref_layer_scope",
            "ba_sa_roi_grid_size",
            "ba_sa_core_ratio",
            "ba_sa_mix_init",
            "spatial_memory_mode",
            "spatial_patch_projection",
            "spatial_kv_init",
            "spatial_kv_kind",
            "spatial_attention_space",
            "spatial_gate_position",
            "spatial_local_window",
            "spatial_delta_rms_cap",
            "total_delta_rms_cap",
            "enable_spatial",
            "enable_identity",
            "scale",
        ):
            if hasattr(processor, field_name):
                value = getattr(processor, field_name)
                if torch.is_tensor(value):
                    value = value.detach().float().cpu().tolist()
                row[field_name] = value
        if getattr(processor, "face_mix_logits", None) is not None:
            row["dual_reference_weight_mean"] = float(
                processor.face_mix_logits.detach().float().sigmoid().mean().item()
            )
        if getattr(processor, "face_residual_gain", None) is not None:
            row["confidence_gain_mean"] = float(
                processor.face_residual_gain.detach().float().tanh().mean().item()
            )
        if getattr(processor, "gate_logit", None) is not None:
            row["effective_spatial_gate"] = float(
                processor.spatial_gate_max
                * torch.sigmoid(processor.gate_logit.detach().float()).item()
            )
        rows.append(row)

    trainable = [
        {
            "name": name,
            "shape": list(parameter.shape),
            "numel": int(parameter.numel()),
        }
        for name, parameter in model.unet.named_parameters()
        if parameter.requires_grad
    ]

    return {
        "processors": rows,
        "self_attention_processor_count": sum(
            row["kind"] == "self" for row in rows
        ),
        "cross_attention_processor_count": sum(
            row["kind"] == "cross" for row in rows
        ),
        "trainable_tensor_count": len(trainable),
        "trainable_parameter_count": sum(
            item["numel"] for item in trainable
        ),
        # Keep the saved model recipe compact. Full parameter state is neither
        # needed nor written by this step-zero search.
        "trainable_parameter_examples": trainable[:50],
    }

def compact_config(cfg, item: ExperimentSpec) -> dict[str, Any]:
    relevant_paths = [
        "disable_branched_sa",
        "disable_branched_ca",
        "strict_face_routing",
        "train_ba_all_steps",
        "train_ba_only",
        "branched_attn_weight_mode",
        "branched_attn_new_weight_kind",
        "train_branched_ca_lora",
        "model.ba_processor_variant",
        "model.ba_site_policy",
        "model.ba_sa_ref_token_mode",
        "model.ba_sa_face_mode",
        "model.ba_sa_ref_layer_scope",
        "model.ba_sa_roi_grid_size",
        "model.ba_sa_core_ratio",
        "model.ba_sa_mix_init",
        "model.ba_target_core_erode_frac",
        "model.ba_output_anchor_mode",
        "model.ba_spatial_memory_mode",
        "model.ba_spatial_patch_projection",
        "model.ba_spatial_patch_dim",
        "model.ba_spatial_kv_init",
        "model.ba_spatial_kv_kind",
        "model.ba_spatial_attention_space",
        "model.ba_spatial_gate_position",
        "model.ba_spatial_mix_mode",
        "model.ba_spatial_local_window",
        "model.ba_spatial_gate_max",
        "model.ba_gate_init_logit",
        "model.ba_spatial_delta_rms_cap",
        "model.ba_total_delta_rms_cap",
        "model.ba_spatial_site_policy",
        "model.ba_identity_token_lane",
        "model.ba_spatial_lane_enabled",
        "model.photomaker_start_step",
        "model.branched_attn_start_step",
        "model.num_inference_steps",
    ]
    values = {}
    for path in relevant_paths:
        value = OmegaConf.select(cfg, path)
        if value is not None:
            values[path] = value
    return {
        "experiment_spec": asdict(item),
        "resolved_relevant_config": values,
        "base_model": BASE_MODEL,
        "photomaker_path": str(PHOTOMAKER_PATH),
        "weight_dtype": MODEL_WEIGHT_DTYPE,
        "rank": MODEL_LORA_RANK,
        "git_commit": GIT_COMMIT,
        "git_branch": GIT_BRANCH,
    }

def build_model_and_pipeline(item: ExperimentSpec):
    cfg = compose_experiment_config(item)

    model = instantiate(
        cfg.model,
        device=DEVICE,
        **ba_runtime_kwargs(cfg),
    )
    model.disable_branched_sa = bool(
        getattr(cfg, "disable_branched_sa", False)
    )
    model.disable_branched_ca = bool(
        getattr(cfg, "disable_branched_ca", False)
    )
    model.strict_face_routing = bool(
        getattr(cfg, "strict_face_routing", False)
    )

    if not hasattr(model, "weight_dtype"):
        raise RuntimeError(
            f"Unsupported model weight dtype: {cfg.model.weight_dtype!r}"
        )

    model.prepare_for_training()
    model.eval()
    model.to(DEVICE)

    pipeline = instantiate(
        cfg.pipeline,
        model=model,
        accelerator=ACCELERATOR,
    )
    pipeline.disable_branched_sa = model.disable_branched_sa
    pipeline.disable_branched_ca = model.disable_branched_ca
    pipeline.strict_face_routing = model.strict_face_routing
    pipeline.to(DEVICE)
    pipeline.id_encoder.to(
        device=DEVICE,
        dtype=model.weight_dtype,
    )
    pipeline.set_progress_bar_config(disable=not SHOW_PROGRESS_BARS)
    pipeline._runtime_uses_branched_unet = None

    if not devices_match(next(pipeline.unet.parameters()).device, DEVICE):
        raise RuntimeError("UNet was not placed on the accelerator device")
    if not devices_match(next(pipeline.id_encoder.parameters()).device, DEVICE):
        raise RuntimeError("PhotoMaker ID encoder was not placed on the accelerator device")

    apply_runtime_mutations(model, pipeline, item.runtime_mutations)
    return cfg, model, pipeline

def cleanup_model(model=None, pipeline=None) -> None:
    try:
        if pipeline is not None:
            reset_branched_generation_caches(pipeline)
    except Exception:
        pass
    del pipeline
    del model
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


In [ ]:
# ---------------------------
# GENERATION HELPERS
# ---------------------------
def seeded_generator(seed: int) -> torch.Generator:
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(int(seed))
    return generator

def diagnostic_summary(pipeline) -> dict[str, Any]:
    records = deepcopy(
        getattr(pipeline, "_ba_ppr_processor_diagnostics", [])
    )
    ratios = [
        float(record["applied_ratio_p50"])
        for record in records
        if record.get("record_type") == "processor_applied_ratio"
        and record.get("applied_ratio_p50") is not None
    ]
    gates = [
        float(record["gate"])
        for record in records
        if record.get("record_type") == "processor_applied_ratio"
        and record.get("gate") is not None
    ]
    return {
        "processor_record_count": len(records),
        "applied_ratio_p50_mean": (
            float(np.mean(ratios)) if ratios else float("nan")
        ),
        "gate_mean": float(np.mean(gates)) if gates else float("nan"),
    }

def generate_sample(
    pipeline,
    sample: dict[str, Any],
    *,
    use_branched_attention: bool,
    item: Optional[ExperimentSpec],
) -> tuple[Image.Image, dict[str, Any]]:
    reset_branched_generation_caches(pipeline)
    pipeline._runtime_uses_branched_unet = None

    ba_start = int(
        (item.generation if item is not None else {}).get(
            "branched_attn_start_step",
            DEFAULT_BA_START_STEP,
        )
    )
    pm_start = int(
        (item.generation if item is not None else {}).get(
            "photomaker_start_step",
            PHOTOMAKER_START_STEP,
        )
    )
    mask_expansion = float(
        (item.generation if item is not None else {}).get(
            "mask_expansion_ratio",
            1.0,
        )
    )
    mask_softness = float(
        (item.generation if item is not None else {}).get(
            "mask_softness",
            0.0,
        )
    )

    if COLLECT_LIGHT_PROCESSOR_DIAGNOSTICS and use_branched_attention:
        pipeline.ba_ppr_collect_diagnostics = True
        pipeline.ba_ppr_diagnostic_steps = (
            ba_start,
            max(ba_start, NUM_INFERENCE_STEPS // 2),
            NUM_INFERENCE_STEPS - 1,
        )
        pipeline.ba_ppr_tensor_diagnostic_sites = ()
        pipeline._ba_ppr_processor_diagnostics = []
        pipeline._ba_ppr_epsilon_diagnostics = []
    else:
        pipeline.ba_ppr_collect_diagnostics = False
        pipeline._ba_ppr_processor_diagnostics = []
        pipeline._ba_ppr_epsilon_diagnostics = []

    reference_image = sample["ref_images"][0]
    kwargs = {
        "prompt": sample["prompt"],
        "negative_prompt": NEGATIVE_PROMPT,
        "input_id_images": [reference_image],
        "height": 1024,
        "width": 1024,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "generator": seeded_generator(sample["seed"]),
        "photomaker_start_step": pm_start,
        "merge_start_step": pm_start,
        "branched_attn_start_step": ba_start,
        "use_branched_attention": bool(use_branched_attention),
        "output_type": "pil",
        "return_dict": True,
        "val_debug": False,
        "debug_dir": None,
    }

    if use_branched_attention:
        kwargs.update(
            {
                "ppr_reference_image": reference_image,
                "ppr_face_bbox_ref": list(sample["face_bbox_ref"]),
                "ppr_reference_noise_seed": REFERENCE_NOISE_SEED,
                "face_bbox_ref": list(sample["face_bbox_ref"]),
                "face_bbox_gen": list(sample["face_bbox_gen"]),
                "auto_mask_ref": False,
                "use_dynamic_mask": False,
                "use_bbox_mask_ref": True,
                "use_bbox_mask_gen": True,
                "mask_expansion_ratio": mask_expansion,
                "mask_softness": mask_softness,
            }
        )

    with torch.inference_mode():
        result = pipeline(**kwargs)
    return result.images[0], diagnostic_summary(pipeline)


## Generate the ordinary PhotoMaker baseline once

Every architecture is compared with the exact same per-sample PM image.

The baseline folder is copied into each experiment folder for self-contained review.


In [ ]:
# ---------------------------
# PM0 BASELINE
# ---------------------------
BASELINE_CONFIG = (
    NN7_V2_BASE
    if (REPO_ROOT / "src/configs/one_id_ba_NN7a_init_v2.yaml").exists()
    else NN7_V1_BASE
)
baseline_spec = spec(
    "pm0_baseline_builder",
    family="baseline",
    role="baseline",
    base_config=BASELINE_CONFIG,
    description="Ordinary PhotoMaker baseline builder; BA disabled.",
)

BASELINE_DIR = EXPERIMENTS_ROOT / f"{RUN_ID}__PM0"
BASELINE_IMAGES_DIR = BASELINE_DIR / "images"
BASELINE_IMAGES_DIR.mkdir(parents=True, exist_ok=False)

baseline_records = []
baseline_images: dict[int, Image.Image] = {}
baseline_face_info: dict[int, dict[str, Any]] = {}
reference_face_info: dict[int, dict[str, Any]] = {}

cfg_base = model_base = pipe_base = None
try:
    cfg_base, model_base, pipe_base = build_model_and_pipeline(
        baseline_spec
    )
    for sample in validation_samples:
        source_index = int(sample["source_index"])
        image, _ = generate_sample(
            pipe_base,
            sample,
            use_branched_attention=False,
            item=None,
        )
        baseline_images[source_index] = image
        baseline_face_info[source_index] = detect_largest_face(image)
        reference_face_info[source_index] = detect_largest_face(
            sample["ref_images"][0]
        )

        image_path = BASELINE_IMAGES_DIR / f"sample_{source_index:02d}_PM0.png"
        image.save(image_path)
        crop_by_bbox(image, sample["face_bbox_gen"]).save(
            BASELINE_IMAGES_DIR
            / f"sample_{source_index:02d}_PM0_face.png"
        )
        sample["ref_images"][0].save(
            BASELINE_IMAGES_DIR
            / f"sample_{source_index:02d}_reference.png"
        )
        baseline_records.append(
            {
                "source_index": source_index,
                "prompt": sample["prompt"],
                "seed": int(sample["seed"]),
                "pm_image": str(image_path),
                "pm_hash": tensor_or_image_hash(image),
                "face_detected": baseline_face_info[source_index]["detected"],
                "reference_path": sample["reference_path"],
            }
        )
finally:
    cleanup_model(model_base, pipe_base)
    assert_core_unchanged()

(BASELINE_DIR / "baseline.json").write_text(
    json.dumps(json_safe(baseline_records), indent=2),
    encoding="utf-8",
)

display(pd.DataFrame(baseline_records))
print("Baseline folder:", BASELINE_DIR)


In [ ]:
# ---------------------------
# EXPERIMENT LOGGING AND DECISION LOGIC
# ---------------------------
def unique_experiment_dir(experiment_id: str) -> Path:
    base = EXPERIMENTS_ROOT / f"{RUN_ID}__{experiment_id}"
    if not base.exists():
        return base
    counter = 2
    while True:
        candidate = EXPERIMENTS_ROOT / f"{RUN_ID}__{experiment_id}__r{counter}"
        if not candidate.exists():
            return candidate
        counter += 1

def aggregate_metrics(rows: list[dict[str, Any]]) -> dict[str, Any]:
    frame = pd.DataFrame(rows)

    def median(name: str) -> float:
        values = pd.to_numeric(frame[name], errors="coerce")
        return float(values.median())

    def mean(name: str) -> float:
        values = pd.to_numeric(frame[name], errors="coerce")
        return float(values.mean())

    detection_rate = float(frame["face_detected"].mean())
    positive_reference_fraction = float(
        (pd.to_numeric(frame["reference_gain_vs_pm"], errors="coerce") > 0).mean()
    )

    aggregate = {
        "sample_count": int(len(frame)),
        "face_detection_rate": detection_rate,
        "median_face_mae_vs_pm": median("face_mae_vs_pm"),
        "median_full_mae_vs_pm": median("full_mae_vs_pm"),
        "median_outside_mae_vs_pm": median("outside_mae_vs_pm"),
        "median_boundary_ring_mae_vs_pm": median("boundary_ring_mae_vs_pm"),
        "median_face_cosine_to_pm_output": median("face_cosine_to_pm_output"),
        "median_reference_gain_vs_pm": median("reference_gain_vs_pm"),
        "mean_reference_gain_vs_pm": mean("reference_gain_vs_pm"),
        "positive_reference_gain_fraction": positive_reference_fraction,
        "median_landmark_displacement_vs_pm": median(
            "landmark_displacement_vs_pm"
        ),
        "median_bbox_iou_vs_pm": median("bbox_iou_vs_pm"),
        "exact_rgb_equal_fraction": float(
            frame["exact_rgb_equal_to_pm"].mean()
        ),
        "median_processor_applied_ratio": median(
            "processor_applied_ratio_p50_mean"
        ),
    }

    visible = (
        aggregate["median_face_mae_vs_pm"] >= FACE_MAE_VISIBLE_MIN
    )
    destructive = (
        aggregate["median_face_mae_vs_pm"]
        > FACE_MAE_DESTRUCTIVE_MAX
    )
    geometry_safe = (
        detection_rate == 1.0
        and aggregate["median_outside_mae_vs_pm"] <= OUTSIDE_MAE_MAX
        and (
            math.isnan(aggregate["median_landmark_displacement_vs_pm"])
            or aggregate["median_landmark_displacement_vs_pm"]
            <= LANDMARK_DISPLACEMENT_MAX
        )
        and (
            math.isnan(aggregate["median_bbox_iou_vs_pm"])
            or aggregate["median_bbox_iou_vs_pm"] >= BBOX_IOU_MIN
        )
    )
    reference_improving = (
        aggregate["median_reference_gain_vs_pm"] >= REFERENCE_GAIN_MIN
        and positive_reference_fraction >= 0.75
    )

    if detection_rate < 1.0:
        decision = "invalid_face_detection"
    elif not visible:
        decision = "too_close_to_photomaker"
    elif destructive or not geometry_safe:
        decision = "n3a_like_unsafe"
    elif reference_improving:
        decision = "promising_step0_candidate"
    else:
        decision = "active_but_not_reference_improving"

    # Only a convenience for sorting. Always inspect the component metrics.
    aggregate["screen_score"] = float(
        10.0 * aggregate["median_reference_gain_vs_pm"]
        + min(aggregate["median_face_mae_vs_pm"], 0.06) / 0.06
        - 4.0 * aggregate["median_outside_mae_vs_pm"]
        - 2.0 * (
            0.0
            if math.isnan(aggregate["median_landmark_displacement_vs_pm"])
            else aggregate["median_landmark_displacement_vs_pm"]
        )
        - 2.0 * max(
            0.0,
            aggregate["median_face_mae_vs_pm"] - 0.08,
        )
    )
    aggregate["visible_change"] = bool(visible)
    aggregate["destructive_change"] = bool(destructive)
    aggregate["geometry_safe"] = bool(geometry_safe)
    aggregate["reference_improving"] = bool(reference_improving)
    aggregate["decision"] = decision
    return aggregate

def processor_applied_ratio(diagnostics: dict[str, Any]) -> float:
    value = diagnostics.get("applied_ratio_p50_mean")
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")

def write_contact_sheet(
    experiment_dir: Path,
    ba_images: dict[int, Image.Image],
) -> Path:
    rows = len(validation_samples)
    columns = 6
    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, 3.4 * rows),
        squeeze=False,
    )

    for row_index, sample in enumerate(validation_samples):
        source_index = int(sample["source_index"])
        reference = sample["ref_images"][0]
        pm = baseline_images[source_index]
        ba = ba_images[source_index]
        bbox = sample["face_bbox_gen"]

        difference = np.abs(image_array(ba) - image_array(pm)).mean(axis=2)

        panels = [
            ("Reference", reference),
            ("PM0", pm),
            ("BA", ba),
            ("PM face", crop_by_bbox(pm, bbox)),
            ("BA face", crop_by_bbox(ba, bbox)),
            ("|BA-PM|", difference),
        ]
        for column_index, (title, value) in enumerate(panels):
            axes[row_index, column_index].imshow(value)
            axes[row_index, column_index].set_title(
                f"{title}\nsample {source_index}"
            )
            axes[row_index, column_index].axis("off")

    plt.tight_layout()
    path = experiment_dir / "contact_sheet.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return path

def write_experiment_readme(
    experiment_dir: Path,
    item: ExperimentSpec,
    aggregate: dict[str, Any],
) -> None:
    text = f"""# {item.experiment_id}

**Family:** {item.family}  
**Role:** {item.role}  
**Decision:** `{aggregate['decision']}`

{item.description}

## Aggregate screen

```json
{json.dumps(json_safe(aggregate), indent=2)}
```

## Files

- `experiment_spec.json` — exact notebook recipe
- `compact_config.json` — resolved relevant Hydra values
- `architecture_signature.json` — installed processor topology
- `metrics_per_sample.csv/json`
- `metrics_summary.json`
- `contact_sheet.png`
- `images/` — reference, PM, BA, fixed face crops and difference maps
"""
    (experiment_dir / "README.md").write_text(text, encoding="utf-8")

def update_registry(record: dict[str, Any]) -> None:
    registry_path = EXPERIMENTS_ROOT / "registry.jsonl"
    with registry_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(json_safe(record)) + "\n")

    rows = []
    for metrics_path in sorted(
        EXPERIMENTS_ROOT.glob("*/metrics_summary.json")
    ):
        try:
            payload = json.loads(metrics_path.read_text(encoding="utf-8"))
            rows.append(payload)
        except Exception:
            continue
    if rows:
        leaderboard = pd.DataFrame(rows)
        order = {
            "promising_step0_candidate": 0,
            "active_but_not_reference_improving": 1,
            "too_close_to_photomaker": 2,
            "n3a_like_unsafe": 3,
            "invalid_face_detection": 4,
            "error": 5,
        }
        leaderboard["_decision_order"] = leaderboard["decision"].map(
            order
        ).fillna(99)
        leaderboard = leaderboard.sort_values(
            ["_decision_order", "screen_score"],
            ascending=[True, False],
        ).drop(columns=["_decision_order"])
        leaderboard.to_csv(
            EXPERIMENTS_ROOT / "leaderboard.csv",
            index=False,
        )

def run_experiment(item: ExperimentSpec) -> dict[str, Any]:
    experiment_dir = unique_experiment_dir(item.experiment_id)
    images_dir = experiment_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=False)

    started = time.time()
    cfg = model = pipeline = None
    rows: list[dict[str, Any]] = []
    ba_images: dict[int, Image.Image] = {}

    spec_payload = {
        **asdict(item),
        "run_id": RUN_ID,
        "git_commit": GIT_COMMIT,
        "git_branch": GIT_BRANCH,
        "sample_indices": SAMPLE_INDICES,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "photomaker_start_step": PHOTOMAKER_START_STEP,
        "default_ba_start_step": DEFAULT_BA_START_STEP,
    }
    (experiment_dir / "experiment_spec.json").write_text(
        json.dumps(json_safe(spec_payload), indent=2),
        encoding="utf-8",
    )

    try:
        cfg, model, pipeline = build_model_and_pipeline(item)

        (experiment_dir / "compact_config.json").write_text(
            json.dumps(
                json_safe(compact_config(cfg, item)),
                indent=2,
            ),
            encoding="utf-8",
        )
        signature = architecture_signature(model, pipeline)
        (experiment_dir / "architecture_signature.json").write_text(
            json.dumps(json_safe(signature), indent=2),
            encoding="utf-8",
        )

        for sample in validation_samples:
            source_index = int(sample["source_index"])
            print(
                f"[{item.experiment_id}] sample {source_index}: "
                f"{sample['prompt'][:60]}"
            )
            ba_image, diagnostics = generate_sample(
                pipeline,
                sample,
                use_branched_attention=True,
                item=item,
            )
            ba_images[source_index] = ba_image

            pm_image = baseline_images[source_index]
            pm_face = baseline_face_info[source_index]
            ba_face = detect_largest_face(ba_image)
            ref_face = reference_face_info[source_index]

            region = region_metrics(
                ba_image,
                pm_image,
                sample["face_bbox_gen"],
            )
            sim_to_pm_output = cosine_np(
                ba_face["embedding"],
                pm_face["embedding"],
            )
            sim_to_reference = cosine_np(
                ba_face["embedding"],
                ref_face["embedding"],
            )
            pm_sim_to_reference = cosine_np(
                pm_face["embedding"],
                ref_face["embedding"],
            )

            row = {
                "experiment_id": item.experiment_id,
                "source_index": source_index,
                "id": sample["id"],
                "prompt": sample["prompt"],
                "seed": int(sample["seed"]),
                **region,
                "face_detected": bool(ba_face["detected"]),
                "pm_face_detected": bool(pm_face["detected"]),
                "face_cosine_to_pm_output": sim_to_pm_output,
                "face_embedding_distance_from_pm": (
                    1.0 - sim_to_pm_output
                    if math.isfinite(sim_to_pm_output)
                    else float("nan")
                ),
                "sim_to_reference": sim_to_reference,
                "pm_sim_to_reference": pm_sim_to_reference,
                "reference_gain_vs_pm": (
                    sim_to_reference - pm_sim_to_reference
                    if (
                        math.isfinite(sim_to_reference)
                        and math.isfinite(pm_sim_to_reference)
                    )
                    else float("nan")
                ),
                "landmark_displacement_vs_pm": (
                    normalized_landmark_displacement(ba_face, pm_face)
                ),
                "bbox_iou_vs_pm": bbox_iou(
                    ba_face["bbox"],
                    pm_face["bbox"],
                ),
                "ba_det_score": ba_face["det_score"],
                "pm_det_score": pm_face["det_score"],
                "processor_applied_ratio_p50_mean": (
                    processor_applied_ratio(diagnostics)
                ),
                "ba_hash": tensor_or_image_hash(ba_image),
                "pm_hash": tensor_or_image_hash(pm_image),
            }
            rows.append(row)

            # Self-contained image bundle.
            reference_path = images_dir / f"sample_{source_index:02d}_reference.png"
            pm_path = images_dir / f"sample_{source_index:02d}_PM0.png"
            ba_path = images_dir / f"sample_{source_index:02d}_BA.png"
            pm_face_path = images_dir / f"sample_{source_index:02d}_PM0_face.png"
            ba_face_path = images_dir / f"sample_{source_index:02d}_BA_face.png"
            diff_path = images_dir / f"sample_{source_index:02d}_diff.png"

            sample["ref_images"][0].save(reference_path)
            pm_image.save(pm_path)
            ba_image.save(ba_path)
            crop_by_bbox(pm_image, sample["face_bbox_gen"]).save(pm_face_path)
            crop_by_bbox(ba_image, sample["face_bbox_gen"]).save(ba_face_path)

            difference = np.abs(
                image_array(ba_image) - image_array(pm_image)
            ).mean(axis=2)
            plt.imsave(diff_path, difference)

        frame = pd.DataFrame(rows)
        aggregate = aggregate_metrics(rows)
        elapsed = time.time() - started
        summary = {
            "run_id": RUN_ID,
            "experiment_id": item.experiment_id,
            "family": item.family,
            "role": item.role,
            "description": item.description,
            "decision": aggregate["decision"],
            "screen_score": aggregate["screen_score"],
            "elapsed_seconds": elapsed,
            "experiment_dir": str(experiment_dir),
            **aggregate,
        }

        frame.to_csv(
            experiment_dir / "metrics_per_sample.csv",
            index=False,
        )
        (experiment_dir / "metrics_per_sample.json").write_text(
            json.dumps(
                json_safe(frame.to_dict(orient="records")),
                indent=2,
            ),
            encoding="utf-8",
        )
        (experiment_dir / "metrics_summary.json").write_text(
            json.dumps(json_safe(summary), indent=2),
            encoding="utf-8",
        )
        write_contact_sheet(experiment_dir, ba_images)
        write_experiment_readme(
            experiment_dir,
            item,
            aggregate,
        )
        update_registry(summary)
        return summary

    except Exception as error:
        error_payload = {
            "run_id": RUN_ID,
            "experiment_id": item.experiment_id,
            "family": item.family,
            "role": item.role,
            "description": item.description,
            "decision": "error",
            "screen_score": float("-inf"),
            "elapsed_seconds": time.time() - started,
            "experiment_dir": str(experiment_dir),
            "error_type": type(error).__name__,
            "error": str(error),
        }
        (experiment_dir / "error.txt").write_text(
            traceback.format_exc(),
            encoding="utf-8",
        )
        (experiment_dir / "metrics_summary.json").write_text(
            json.dumps(json_safe(error_payload), indent=2),
            encoding="utf-8",
        )
        update_registry(error_payload)
        print(traceback.format_exc())
        return error_payload

    finally:
        cleanup_model(model, pipeline)
        assert_core_unchanged()


## Run the selected architecture suite

The full quick suite can take substantial GPU time because each architecture is rebuilt and four 1024×1024 images are generated.

For short iterations, set `SELECTED_EXPERIMENT_IDS` in the first cell.


In [ ]:
# ---------------------------
# RUN SEARCH
# ---------------------------
search_results = []

for experiment_number, item in enumerate(EXPERIMENT_SPECS, start=1):
    print(
        "\n"
        + "=" * 100
        + f"\n[{experiment_number}/{len(EXPERIMENT_SPECS)}] "
        + item.experiment_id
        + "\n"
        + item.description
        + "\n"
        + "=" * 100
    )
    result = run_experiment(item)
    search_results.append(result)
    display(pd.DataFrame([result]))

results_df = pd.DataFrame(search_results)
display(
    results_df.sort_values(
        ["decision", "screen_score"],
        ascending=[True, False],
    )
)

print("Master leaderboard:", EXPERIMENTS_ROOT / "leaderboard.csv")


In [ ]:
# ---------------------------
# REVIEW CURRENT LEADERBOARD
# ---------------------------
leaderboard_path = EXPERIMENTS_ROOT / "leaderboard.csv"
if leaderboard_path.exists():
    leaderboard = pd.read_csv(leaderboard_path)
    preferred_columns = [
        "experiment_id",
        "family",
        "role",
        "decision",
        "screen_score",
        "median_face_mae_vs_pm",
        "median_face_cosine_to_pm_output",
        "median_reference_gain_vs_pm",
        "positive_reference_gain_fraction",
        "median_landmark_displacement_vs_pm",
        "median_bbox_iou_vs_pm",
        "median_outside_mae_vs_pm",
        "face_detection_rate",
        "experiment_dir",
    ]
    display(
        leaderboard[
            [column for column in preferred_columns if column in leaderboard]
        ]
    )
else:
    print("No leaderboard exists yet.")


## How to iterate without touching core code

Edit only `EXPERIMENT_SPECS` or append a new `ExperimentSpec` in this notebook.

### Highest-value N3a-like search axes

1. **Reference ownership**
   - `dual`: `ba_sa_mix_init = 0.35, 0.50, 0.65, 0.75, 0.85`
   - `core_ring`: `ba_sa_core_ratio = 0.45–0.85`
   - `confidence_residual`: in-memory gain `0.25–0.75`
   - `reference`: 100% reference ownership, positive control only

2. **Reference memory**
   - `full_grid`: closest to N3a, but retains zero positions and absolute layout
   - `roi`: normalized dense ROI, preferred repair
   - `ba_sa_roi_grid_size = 6, 8, 12, 16`

3. **Layer scope**
   - `all`: strongest and most N3a-like, highest geometry risk
   - `up`: protects down/mid geometry while retaining spatial reference attention

4. **Boundary and output ownership**
   - `core_ring` keeps a target-owned face boundary
   - `ba_output_anchor_mode = base_outside_core`
   - `ba_target_core_erode_frac = 0.05–0.20`

5. **Cross-attention**
   - test exact N3a with CA on once
   - keep CA off for serious candidates; historical results show trainable split CA causes collapse

6. **Timing**
   - 20-step screen: BA starts at `4, 6, 8, 10`
   - corresponding 50-step runs: `10, 15, 20, 25`
   - earlier gives stronger reference ownership and more pose risk
   - later gives safer but weaker changes

7. **Routing**
   - `strict_face_routing = false` preserves more target continuity
   - test `true` only as an ablation; it can remove useful target residual information

8. **Modern clean-patch controls**
   - initial alpha `0.10–0.50`
   - local window `5, 9, 15`
   - up1 versus all up blocks
   - cap `0.20–0.50`
   - these are comparison controls; the main search should remain on target-Q → spatial-reference-K/V

### Do not promote a config merely because it differs from PhotoMaker

The desired step-zero region is:

```text
visible face change
+ face detection 4/4
+ positive movement toward the reference identity
+ stable landmarks and detected bbox
+ small outside-face and boundary-ring change
```

N3a is expected to pass the first line and fail several others.

### Promotion sequence

1. Run the 20-step first-four screen.
2. Select at most three candidates classified as `promising_step0_candidate`.
3. Visually inspect every crop and difference map.
4. Repeat the finalists at 50 steps on the same four records.
5. Run a wrong-reference A→B control while keeping PM identity, prompt, seed and bbox fixed.
6. Expand to the 24-case deterministic matrix.
7. Only then create a training config.

### Suggested interpretation

| Decision | Meaning |
|---|---|
| `too_close_to_photomaker` | Step-zero branch authority is not visually meaningful |
| `active_but_not_reference_improving` | The face changes, but not toward the supplied identity |
| `n3a_like_unsafe` | Strong branch, but geometry/outside-face damage is too high |
| `promising_step0_candidate` | Visible, reference-improving and provisionally geometry-safe |
| `invalid_face_detection` | Reject |
| `error` | Fix experiment recipe before interpreting |

The four-image screen is a search tool, not evidence of generalization.


In [ ]:
# ---------------------------
# FINAL CORE-CODE CHECK
# ---------------------------
assert_core_unchanged()
print("Protected core code is unchanged.")
print("Artifacts were written only under:", EXPERIMENTS_ROOT)
print("Git status now:\n", git_text("status", "--porcelain"))
